# T2.4 – DBRepo View Creation
**Project:** Predicting the Market Value of Football Players  
**Owner:** Student D  
**Database ID:** `598ce585-d8b5-4a97-8f19-cb085d4a5b1e`

## Purpose
Creates views in DBRepo that expose query-ready subsets of the 3NF schema for
the ML pipeline. Also retrieves and merges the denormalised datasets via the
DBRepo REST API — replacing all local CSV/Excel reads in the ML pipeline (T2.6).

## Views created
| View | Table | Purpose |
|---|---|---|
| `vw_forward_features` | `forward_player_valuation` | Dataset 1 ML features |
| `vw_transfer_features` | `transfer_value_observation` | Dataset 2 ML features (main) |
| `vw_combined_player_value` | `forward_player_valuation` | Cross-dataset exploration |
| `vw_player_lookup` | `player` | player_id → player_name |
| `vw_club_lookup` | `club` | club_id → club_name |
| `vw_position_lookup` | `position` | position_id → position_name |
| `vw_nationality_lookup` | `nationality` | nationality_id → nationality_name |

## Why no JOIN views?
The DBRepo SDK mapper internally calls `numpy.array()` on each involved
table's column list. When tables have different column counts the array
becomes inhomogeneous and raises `ValueError`. The workaround is to create
fact-table views without joins, then merge lookup data locally in pandas
after retrieving each view via the REST API.

## 1. Imports

In [2]:
import pandas as pd
from getpass import getpass
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition

## 2. Configuration

In [3]:
DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME        = "ekene"

# New database — ekene is full owner
DATABASE_ID     = "598ce585-d8b5-4a97-8f19-cb085d4a5b1e"

# ── View names ─────────────────────────────────────────────────────────────────
VIEW_FORWARD     = "vw_forward_features"
VIEW_TRANSFER    = "vw_transfer_features"
VIEW_COMBINED    = "vw_combined_player_value"
VIEW_PLAYER      = "vw_player_lookup"
VIEW_CLUB        = "vw_club_lookup"
VIEW_POSITION    = "vw_position_lookup"
VIEW_NATIONALITY = "vw_nationality_lookup"

ALL_VIEW_NAMES = [
    VIEW_FORWARD, VIEW_TRANSFER, VIEW_COMBINED,
    VIEW_PLAYER, VIEW_CLUB, VIEW_POSITION, VIEW_NATIONALITY,
]

## 3. Connect

In [4]:
password = getpass(f"DBRepo password for '{USERNAME}': ")
client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=USERNAME,
    password=password,
)
print(f"Connected as: {client.whoami()}")

DBRepo password for 'ekene':  ········


ekene
Connected as: ekene


## 4. Fetch Table Metadata

In [5]:
# get_tables() returns TableBrief (no column info) — must call get_table() per table
all_tables_brief = client.get_tables(database_id=DATABASE_ID)
table_by_name = {}
for t in all_tables_brief:
    full = client.get_table(database_id=DATABASE_ID, table_id=t.id)
    table_by_name[full.name] = full

print("Tables in database:")
for name, tbl in table_by_name.items():
    cols = [c.internal_name for c in tbl.columns]
    print(f"  {name} ({len(cols)} cols): {cols}")

Tables in database:
  source_dataset (11 cols): ['source_dataset_id', 'dataset_title', 'original_creators', 'publisher', 'version_label', 'doi', 'doi_url', 'license_name', 'license_url', 'source_filename', 'notes']
  transfer_value_observation (28 cols): ['transfer_observation_id', 'source_dataset_id', 'player_id', 'position_id', 'nationality_id', 'club_id', 'season_year', 'original_player_name', 'original_position_name', 'original_nationality_name', 'original_club_name', 'age_then_years', 'age_now_years', 'club_performance', 'relegation', 'success_or_not', 'total_games', 'assists', 'penalty_kicks', 'total_minutes', 'total_goals', 'height_cm', 'start_value_eur', 'end_value_eur', 'delta_value_eur', 'value_start_mln', 'value_end_mln', 'value_delta_mln']
  forward_player_valuation (16 cols): ['forward_valuation_id', 'source_dataset_id', 'player_id', 'club_id', 'original_player_name', 'original_team_name', 'player_age_years', 'market_value_mln', 'value_rank', 'plays_in_europe', 'matches_pl

## 5. Helper Functions

In [6]:
def get_columns(table_by_name, table_name, exclude=None):
    """
    Returns list of 'table.column' strings for a table, excluding specified columns.
    """
    exclude = exclude or []
    tbl = table_by_name[table_name]
    return [
        f"{table_name}.{col.internal_name}"
        for col in tbl.columns
        if col.internal_name not in exclude
    ]


def create_view_safe(client, database_id, name, query):
    """
    Creates a view if it does not already exist. Safe to re-run.
    Returns the view id.
    """
    existing = {v.name: v.id for v in client.get_views(database_id=database_id)}
    if name in existing:
        print(f"[SKIP] '{name}' already exists (id: {existing[name]})")
        return existing[name]
    try:
        view = client.create_view(
            database_id=database_id,
            name=name,
            query=query,
            is_public=True,
            is_schema_public=True,
        )
        print(f"[OK]   '{name}' created (id: {view.id})")
        return view.id
    except Exception as e:
        print(f"[FAIL] '{name}': {type(e).__name__}: {e}")
        return None


def fetch_view_df(client, database_id, view_id_map, view_name):
    """
    Fetches view data from DBRepo REST API and returns a pandas DataFrame.
    Returns empty DataFrame if view not found or data not yet loaded.
    """
    if view_name not in view_id_map:
        print(f"[MISSING] {view_name}")
        return pd.DataFrame()
    try:
        df = client.get_view_data(
            database_id=database_id,
            view_id=view_id_map[view_name],
        )
        print(f"[OK]   {view_name}: {len(df)} rows, {len(df.columns)} cols")
        return df
    except Exception as e:
        print(f"[FAIL] {view_name}: {e}")
        return pd.DataFrame()


print("Helpers defined.")

Helpers defined.


## 6. Delete Existing Views (optional — run to reset)
Only run this cell if you need to recreate views from scratch.

In [7]:
# Uncomment and run this cell ONLY if you want to delete and recreate all views

# existing_views = client.get_views(database_id=DATABASE_ID)
# if not existing_views:
#     print("No views to delete.")
# for v in existing_views:
#     try:
#         client.delete_view(database_id=DATABASE_ID, view_id=v.id)
#         print(f"Deleted: {v.name}")
#     except Exception as e:
#         print(f"Failed to delete {v.name}: {e}")

print("(Delete cell is commented out — uncomment to reset views)")

(Delete cell is commented out — uncomment to reset views)


## 7. View 1 — `vw_forward_features`

**Purpose:** Exposes all numerical and categorical features from Dataset 1
(Forward football player valuation) for ML training.  
**Target variable:** `market_value_mln`  
**Excludes:** `original_player_name`, `original_team_name` (duplicates resolved via lookup views)

**Equivalent SQL:**
```sql
SELECT forward_valuation_id, source_dataset_id, player_id, club_id,
       player_age_years, market_value_mln, value_rank, plays_in_europe,
       matches_played, goals, assists, minutes_per_goal, minutes_played,
       instagram_followers_mln
FROM forward_player_valuation;
```

In [8]:
forward_cols = get_columns(
    table_by_name,
    table_name='forward_player_valuation',
    exclude=[
        'original_player_name',  # duplicate — use vw_player_lookup for names
        'original_team_name',    # duplicate — use vw_club_lookup for names
    ],
)
print("Columns for vw_forward_features:")
for c in forward_cols: print(f"  {c}")

query_forward = QueryDefinition(
    datasources=['forward_player_valuation'],
    columns=forward_cols,
    joins=[],
)

view_forward_id = create_view_safe(
    client=client,
    database_id=DATABASE_ID,
    name=VIEW_FORWARD,
    query=query_forward,
)

Columns for vw_forward_features:
  forward_player_valuation.forward_valuation_id
  forward_player_valuation.source_dataset_id
  forward_player_valuation.player_id
  forward_player_valuation.club_id
  forward_player_valuation.player_age_years
  forward_player_valuation.market_value_mln
  forward_player_valuation.value_rank
  forward_player_valuation.plays_in_europe
  forward_player_valuation.matches_played
  forward_player_valuation.goals
  forward_player_valuation.assists
  forward_player_valuation.minutes_per_goal
  forward_player_valuation.minutes_played
  forward_player_valuation.instagram_followers_mln
[OK]   'vw_forward_features' created (id: 6134001d-73e9-4b25-b4f3-1d66bd663c50)


## 8. View 2 — `vw_transfer_features`

**Purpose:** Exposes all performance and valuation features from Dataset 2
(Transfer Value Determinants) across seasons 2019-2023. This is the **primary
ML training dataset** used by the XGBoost regression model in `Project.ipynb`.  
**Target variable:** `value_end_mln`  
**Excludes:** original name columns (use lookup views), raw EUR value columns
(use the `_mln` equivalents which are already in consistent units)

**ML features used by the model (from Project.ipynb):**
`position_id`, `nationality_id`, `club_id`, `age_then_years`, `age_now_years`,
`total_games`, `assists`, `penalty_kicks`, `total_minutes`, `total_goals`,
`height_cm`, `value_start_mln`, `season_year`

**Equivalent SQL:**
```sql
SELECT transfer_observation_id, source_dataset_id, player_id,
       position_id, nationality_id, club_id, season_year,
       age_then_years, age_now_years, club_performance, relegation,
       success_or_not, total_games, assists, penalty_kicks,
       total_minutes, total_goals, height_cm,
       value_start_mln, value_end_mln, value_delta_mln
FROM transfer_value_observation;
```

In [9]:
transfer_cols = get_columns(
    table_by_name,
    table_name='transfer_value_observation',
    exclude=[
        'original_player_name',      # use vw_player_lookup
        'original_position_name',    # use vw_position_lookup
        'original_nationality_name', # use vw_nationality_lookup
        'original_club_name',        # use vw_club_lookup
        # Exclude raw EUR columns — _mln versions are in consistent units
        # and are what the ML pipeline uses
        'start_value_eur',
        'end_value_eur',
        'delta_value_eur',
    ],
)
print("Columns for vw_transfer_features:")
for c in transfer_cols: print(f"  {c}")

query_transfer = QueryDefinition(
    datasources=['transfer_value_observation'],
    columns=transfer_cols,
    joins=[],
)

view_transfer_id = create_view_safe(
    client=client,
    database_id=DATABASE_ID,
    name=VIEW_TRANSFER,
    query=query_transfer,
)

Columns for vw_transfer_features:
  transfer_value_observation.transfer_observation_id
  transfer_value_observation.source_dataset_id
  transfer_value_observation.player_id
  transfer_value_observation.position_id
  transfer_value_observation.nationality_id
  transfer_value_observation.club_id
  transfer_value_observation.season_year
  transfer_value_observation.age_then_years
  transfer_value_observation.age_now_years
  transfer_value_observation.club_performance
  transfer_value_observation.relegation
  transfer_value_observation.success_or_not
  transfer_value_observation.total_games
  transfer_value_observation.assists
  transfer_value_observation.penalty_kicks
  transfer_value_observation.total_minutes
  transfer_value_observation.total_goals
  transfer_value_observation.height_cm
  transfer_value_observation.value_start_mln
  transfer_value_observation.value_end_mln
  transfer_value_observation.value_delta_mln
[OK]   'vw_transfer_features' created (id: 436a5cca-8f20-4661-9d3f-626

## 9. View 3 — `vw_combined_player_value`

**Purpose:** Shared-schema view over Dataset 1 exposing only the columns that
have a direct equivalent in Dataset 2. Useful for cross-dataset exploratory
analysis comparing feature distributions across both sources.

**Note:** A true UNION of both datasets is not supported by the DBRepo SDK.
The full UNION SQL is documented below for reference.

**Equivalent SQL (reference — not executable in DBRepo SDK):**
```sql
SELECT player_id, club_id, player_age_years AS age_years,
       goals, assists, minutes_played, market_value_mln, 'dataset_1' AS source
FROM forward_player_valuation
UNION ALL
SELECT player_id, club_id, age_then_years AS age_years,
       total_goals AS goals, assists, total_minutes AS minutes_played,
       value_end_mln AS market_value_mln, 'dataset_2' AS source
FROM transfer_value_observation;
```

In [10]:
# Only include the columns that have equivalents in both datasets
combined_cols = get_columns(
    table_by_name,
    table_name='forward_player_valuation',
    exclude=[
        'original_player_name',
        'original_team_name',
        'source_dataset_id',
        'value_rank',
        'plays_in_europe',       # no equivalent in Dataset 2
        'matches_played',        # Dataset 2 uses total_games
        'minutes_per_goal',      # derived stat, no equivalent
        'instagram_followers_mln', # no equivalent in Dataset 2
        # Kept: player_id, club_id, player_age_years, goals,
        #       assists, minutes_played, market_value_mln (TARGET)
    ],
)
print("Columns for vw_combined_player_value:")
for c in combined_cols: print(f"  {c}")

query_combined = QueryDefinition(
    datasources=['forward_player_valuation'],
    columns=combined_cols,
    joins=[],
)

view_combined_id = create_view_safe(
    client=client,
    database_id=DATABASE_ID,
    name=VIEW_COMBINED,
    query=query_combined,
)

Columns for vw_combined_player_value:
  forward_player_valuation.forward_valuation_id
  forward_player_valuation.player_id
  forward_player_valuation.club_id
  forward_player_valuation.player_age_years
  forward_player_valuation.market_value_mln
  forward_player_valuation.goals
  forward_player_valuation.assists
  forward_player_valuation.minutes_played
[OK]   'vw_combined_player_value' created (id: 0c834912-d507-442a-876e-4dc79dd3e079)


## 10. Lookup Views
Lightweight views exposing lookup tables. Merged locally with fact views
in pandas to reconstruct denormalised datasets for the ML pipeline.

In [11]:
def create_lookup_view(client, database_id, view_name, table_name, table_by_name):
    """Create a simple single-table view with all columns."""
    cols = get_columns(table_by_name, table_name)
    query = QueryDefinition(
        datasources=[table_name],
        columns=cols,
        joins=[],
    )
    return create_view_safe(client, database_id, view_name, query)


# ── player: player_id → player_name ───────────────────────────────────────────
create_lookup_view(client, DATABASE_ID, VIEW_PLAYER, 'player', table_by_name)

# ── club: club_id → club_name ─────────────────────────────────────────────────
create_lookup_view(client, DATABASE_ID, VIEW_CLUB, 'club', table_by_name)

# ── position: position_id → position_name ─────────────────────────────────────
create_lookup_view(client, DATABASE_ID, VIEW_POSITION, 'position', table_by_name)

# ── nationality: nationality_id → nationality_name ────────────────────────────
create_lookup_view(client, DATABASE_ID, VIEW_NATIONALITY, 'nationality', table_by_name)

[OK]   'vw_player_lookup' created (id: 3fe7da55-c3e7-4859-a0a1-2e55759b4602)
[OK]   'vw_club_lookup' created (id: 2b80b0ff-aaf3-440e-a8af-ceb226dca11c)
[OK]   'vw_position_lookup' created (id: 1b0fce54-b1cd-4709-a2d9-8602dd316144)
[OK]   'vw_nationality_lookup' created (id: 1554d194-a3cc-49e4-a250-dcc530e0110d)


'1554d194-a3cc-49e4-a250-dcc530e0110d'

## 11. Verify All Views

In [12]:
all_views   = client.get_views(database_id=DATABASE_ID)
view_id_map = {v.name: v.id for v in all_views}

print(f"Views in database '{DATABASE_ID}':")
print("-" * 60)
for v in all_views:
    print(f"  {v.name}: {v.id}")

missing = set(ALL_VIEW_NAMES) - set(view_id_map)
print()
if missing:
    print(f"WARNING — missing views: {missing}")
else:
    print("All 7 views registered successfully.")

Views in database '598ce585-d8b5-4a97-8f19-cb085d4a5b1e':
------------------------------------------------------------
  vw_nationality_lookup: 1554d194-a3cc-49e4-a250-dcc530e0110d
  vw_position_lookup: 1b0fce54-b1cd-4709-a2d9-8602dd316144
  vw_club_lookup: 2b80b0ff-aaf3-440e-a8af-ceb226dca11c
  vw_player_lookup: 3fe7da55-c3e7-4859-a0a1-2e55759b4602
  vw_combined_player_value: 0c834912-d507-442a-876e-4dc79dd3e079
  vw_transfer_features: 436a5cca-8f20-4661-9d3f-626da84b4362
  vw_forward_features: 6134001d-73e9-4b25-b4f3-1d66bd663c50

All 7 views registered successfully.


## 12. Retrieve and Reconstruct ML Datasets via REST API
**T2.6 requirement:** all data loaded from DBRepo REST API — no local file reads.

Run this section after Student C confirms data is loaded (T2.5).
Empty DataFrames before data is loaded are expected.

In [18]:
# ── Fetch all views ────────────────────────────────────────────────────────────
print("Fetching views from DBRepo API...")
print("-" * 50)

df_forward     = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_FORWARD)
df_transfer    = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_TRANSFER)
df_combined    = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_COMBINED)
df_player      = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_PLAYER)
df_club        = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_CLUB)
df_position    = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_POSITION)
df_nationality = fetch_view_df(client, DATABASE_ID, view_id_map, VIEW_NATIONALITY)

Fetching views from DBRepo API...
--------------------------------------------------
[OK]   vw_forward_features: 438 rows, 14 cols
[OK]   vw_transfer_features: 2502 rows, 21 cols
[OK]   vw_combined_player_value: 438 rows, 8 cols
[OK]   vw_player_lookup: 996 rows, 2 cols
[OK]   vw_club_lookup: 205 rows, 2 cols
[OK]   vw_position_lookup: 13 rows, 2 cols
[OK]   vw_nationality_lookup: 75 rows, 2 cols


In [19]:
# ── Reconstruct Dataset 1: forward player features ────────────────────────────
# Merge player and club names onto the forward fact view
if len(df_forward) > 0 and len(df_player) > 0 and len(df_club) > 0:
    ml_forward = (
        df_forward
        .merge(df_player, on='player_id', how='left')
        .merge(df_club,   on='club_id',   how='left')
    )
    print(f"Dataset 1 (forward): {len(ml_forward)} rows x {len(ml_forward.columns)} cols")
    print(f"  Columns: {ml_forward.columns.tolist()}")
    display(ml_forward.head(3))
else:
    ml_forward = pd.DataFrame()
    print("Dataset 1: no data yet (run after T2.5 data load)")

Dataset 1 (forward): 438 rows x 16 cols
  Columns: ['assists', 'club_id', 'forward_valuation_id', 'goals', 'instagram_followers_mln', 'market_value_mln', 'matches_played', 'minutes_per_goal', 'minutes_played', 'player_age_years', 'player_id', 'plays_in_europe', 'source_dataset_id', 'value_rank', 'player_name', 'club_name']


,assists,club_id,forward_valuation_id,goals,instagram_followers_mln,market_value_mln,matches_played,minutes_per_goal,minutes_played,player_age_years,player_id,plays_in_europe,source_dataset_id,value_rank,player_name,club_name
0,18,166,173,86,0.247,15.0,259,210,12865,25,53,1,1,4,Andrea Pinamonti,Sassuolo
1,56,35,380,106,0.051,8.0,302,203,21547,26,441,1,1,1,Jesper Karlsson,Bolonia
2,2,157,218,10,0.074,12.0,53,328,3280,18,89,1,1,3,Assane Diao,Real Betis


In [20]:
# ── Reconstruct Dataset 2: transfer features ──────────────────────────────────
# This is the PRIMARY ML dataset used by Project.ipynb
# The ML model needs: position, nationality, club_name, age_then, age_now,
# total_games, assists, penalty_kicks, total_minutes, total_goals,
# height_cm, value_start_mln, season_year → predicts value_end_mln

if (len(df_transfer) > 0 and len(df_player) > 0
        and len(df_club) > 0 and len(df_position) > 0
        and len(df_nationality) > 0):

    ml_transfer = (
        df_transfer
        .merge(df_player,      on='player_id',      how='left')
        .merge(df_club,        on='club_id',         how='left')
        .merge(df_position,    on='position_id',     how='left')
        .merge(df_nationality, on='nationality_id',  how='left')
    )
    print(f"Dataset 2 (transfer): {len(ml_transfer)} rows x {len(ml_transfer.columns)} cols")
    print(f"  Columns: {ml_transfer.columns.tolist()}")
    display(ml_transfer.head(3))
else:
    ml_transfer = pd.DataFrame()
    print("Dataset 2: no data yet (run after T2.5 data load)")

Dataset 2 (transfer): 2502 rows x 25 cols
  Columns: ['age_now_years', 'age_then_years', 'assists', 'club_id', 'club_performance', 'height_cm', 'nationality_id', 'penalty_kicks', 'player_id', 'position_id', 'relegation', 'season_year', 'source_dataset_id', 'success_or_not', 'total_games', 'total_goals', 'total_minutes', 'transfer_observation_id', 'value_delta_mln', 'value_end_mln', 'value_start_mln', 'player_name', 'club_name', 'position_name', 'nationality_name']


,age_now_years,age_then_years,assists,club_id,club_performance,height_cm,nationality_id,penalty_kicks,player_id,position_id,...,total_goals,total_minutes,transfer_observation_id,value_delta_mln,value_end_mln,value_start_mln,player_name,club_name,position_name,nationality_name
0,34,29,2,57,-5.0,176.0,48,1,759,9,...,3,2507,91,-2.5,9.5,12.0,Patrick van Aanholt,Crystal Palace,Left-Back,Netherlands
1,27,24,1,119,NaN,184.0,48,0,236,2,...,1,526,1609,-14.0,30.0,44.0,Donny van de Beek,Manchester United,Central Midfield,Netherlands
2,35,32,3,199,NaN,170.0,25,0,2,9,...,2,2726,1694,-1.0,3.0,4.0,Aaron Cresswell,West Ham United,Left-Back,England


In [21]:
# ── Reconstruct combined dataset for exploratory analysis ─────────────────────
if len(df_combined) > 0 and len(df_player) > 0 and len(df_club) > 0:
    ml_combined = (
        df_combined
        .merge(df_player, on='player_id', how='left')
        .merge(df_club,   on='club_id',   how='left')
    )
    print(f"Combined dataset: {len(ml_combined)} rows x {len(ml_combined.columns)} cols")
    display(ml_combined.head(3))
else:
    ml_combined = pd.DataFrame()
    print("Combined: no data yet (run after T2.5 data load)")

Combined dataset: 438 rows x 10 cols


,assists,club_id,forward_valuation_id,goals,market_value_mln,minutes_played,player_age_years,player_id,player_name,club_name
0,34,205,103,51,20.0,3706,22,182,Crysencio Summerville,west haam
1,1,203,258,8,11.0,349,25,599,Maksim Glushenkov,Zenit
2,11,138,157,13,15.0,5036,19,601,Malick Fofana,Olimpic Lyon


## 13. Column Mapping — DBRepo → Project.ipynb

The ML pipeline in `Project.ipynb` reads from local Excel files using these
original column names. After loading from DBRepo the columns are renamed
to match so `Project.ipynb` can run without modification.

| Project.ipynb column | DBRepo column (ml_transfer) |
|---|---|
| `position` | `position_name` |
| `nationality` | `nationality_name` |
| `club_name` | `club_name` |
| `age_then` | `age_then_years` |
| `age_now` | `age_now_years` |
| `height` | `height_cm` |
| `start_value` | `value_start_mln` × 1,000,000 |
| `value_0_mln` | `value_start_mln` |
| `season` | `season_year` |
| `player_name` | `player_name` |
| `value_end_mln` | `value_end_mln` (TARGET) |

In [22]:
# Rename DBRepo columns to match Project.ipynb column names
# Run this after Section 12 has populated ml_transfer

if len(ml_transfer) > 0:
    transfer_for_ml = ml_transfer.rename(columns={
        'position_name':   'position',
        'nationality_name':'nationality',
        'age_then_years':  'age_then',
        'age_now_years':   'age_now',
        'height_cm':       'height',
        'value_start_mln': 'value_0_mln',
        'season_year':     'season',
    })

    # Project.ipynb uses 'start_value' in EUR — reconstruct from mln column
    transfer_for_ml['start_value'] = transfer_for_ml['value_0_mln'] * 1_000_000

    # Verify the columns Project.ipynb expects are all present
    required_by_project = [
        'position', 'nationality', 'club_name', 'age_then', 'age_now',
        'total_games', 'assists', 'penalty_kicks', 'total_minutes',
        'total_goals', 'height', 'start_value', 'value_0_mln',
        'season', 'value_end_mln', 'player_name',
    ]
    missing_cols = [c for c in required_by_project if c not in transfer_for_ml.columns]

    if missing_cols:
        print(f"WARNING: missing columns for Project.ipynb: {missing_cols}")
    else:
        print(f"transfer_for_ml ready: {len(transfer_for_ml)} rows")
        print("All columns required by Project.ipynb are present.")
        display(transfer_for_ml[required_by_project].head(3))
else:
    transfer_for_ml = pd.DataFrame()
    print("No data yet — run after T2.5 data load")

transfer_for_ml ready: 2502 rows
All columns required by Project.ipynb are present.


,position,nationality,club_name,age_then,age_now,total_games,assists,penalty_kicks,total_minutes,total_goals,height,start_value,value_0_mln,season,value_end_mln,player_name
0,Left-Back,Netherlands,Crystal Palace,29,34,29,2,1,2507,3,176.0,12000000.0,12.0,2019,9.5,Patrick van Aanholt
1,Central Midfield,Netherlands,Manchester United,24,27,19,1,0,526,1,184.0,44000000.0,44.0,2020,30.0,Donny van de Beek
2,Left-Back,England,West Ham United,32,35,31,3,0,2726,2,170.0,4000000.0,4.0,2021,3.0,Aaron Cresswell


## 14. Summary

| View | Table | Columns | Purpose |
|---|---|---|---|
| `vw_forward_features` | `forward_player_valuation` | All except original names | Dataset 1 ML features |
| `vw_transfer_features` | `transfer_value_observation` | All except original names + raw EUR cols | Dataset 2 ML features |
| `vw_combined_player_value` | `forward_player_valuation` | Shared feature subset | Cross-dataset EDA |
| `vw_player_lookup` | `player` | player_id, player_name | ID → name join |
| `vw_club_lookup` | `club` | club_id, club_name | ID → name join |
| `vw_position_lookup` | `position` | position_id, position_name | ID → name join |
| `vw_nationality_lookup` | `nationality` | nationality_id, nationality_name | ID → name join |

